<a href="https://colab.research.google.com/github/Tin123-alt/practice/blob/main/finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = "/content/drive/MyDrive/emotion_phobert_3class"
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}
!pwd

Mounted at /content/drive
/content/drive/MyDrive/emotion_phobert_3class
/content/drive/MyDrive/emotion_phobert_3class


In [3]:
!pip install -q transformers datasets evaluate accelerate py_vncorenlp scikit-learn pandas torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.2 MB/s eta 0:00:00


In [4]:
import os
import py_vncorenlp

MODEL_DIR = "/content/vncorenlp"
os.makedirs(MODEL_DIR, exist_ok=True)

if not os.path.exists(os.path.join(MODEL_DIR, "models")):
    print("Tải VnCoreNLP models lần đầu...")
    py_vncorenlp.download_model(save_dir=MODEL_DIR)

rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=MODEL_DIR)
print("Word segmenter đã sẵn sàng")

Tải VnCoreNLP models lần đầu...
Word segmenter đã sẵn sàng


In [5]:
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
from sklearn.model_selection import train_test_split

ds = load_dataset("tridm/UIT-VSMEC")
df = pd.concat([ds[k].to_pandas() for k in ['train', 'validation', 'test']], ignore_index=True)

emotion_map = {
    'Enjoyment': 'vui vẻ',
    'Anger': 'tức giận',
    'Sadness': 'buồn bã'
}
df['emotion'] = df['Emotion'].map(emotion_map)
df = df.dropna(subset=['emotion']).reset_index(drop=True)
df = df.rename(columns={'Sentence': 'text'})[['text', 'emotion']]

# Word segmentation
df['text'] = df['text'].astype(str).apply(
    lambda x: " ".join(rdrsegmenter.word_segment(x.strip()))
)

# Tạo label
label_map = {"tức giận": 0, "vui vẻ": 1, "buồn bã": 2}
df['label'] = df['emotion'].map(label_map)

# Split
train_df, temp = train_test_split(df, test_size=0.25, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp, test_size=0.5, stratify=temp['label'], random_state=42)

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df[['text', 'label']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))
})

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

valid.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5548 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/686 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/693 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2695
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 449
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 450
    })
})


In [6]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

# Tạo DatasetDict
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['text', 'label']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df[['text', 'label']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df[['text', 'label']].reset_index(drop=True))
})

print(dataset)

# Tokenization
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
        return_tensors="pt"
    )

tokenized_datasets = dataset.map(preprocess_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

print("Tokenization hoàn tất. Cấu trúc:", tokenized_datasets)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 2695
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 449
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 450
    })
})


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2695 [00:00<?, ? examples/s]

Map:   0%|          | 0/449 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenization hoàn tất. Cấu trúc: DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2695
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 449
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 450
    })
})


In [15]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import numpy as np

train_labels = np.array(tokenized_datasets["train"]["labels"])

unique_classes = np.unique(train_labels)
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=unique_classes,
    y=train_labels
)

class_weights = torch.tensor(class_weights_np, dtype=torch.float)
if torch.cuda.is_available():
    class_weights = class_weights.cuda()

print(f"Class weights cho các lớp {unique_classes.tolist()}: {class_weights.tolist()}")

Class weights cho các lớp [0, 1, 2]: [2.495370388031006, 0.6098664999008179, 1.0421500205993652]


In [16]:
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        # Forward
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # Weighted Cross Entropy
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [17]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./phobert-3emotion-checkpoints",     #checkpoint
    num_train_epochs=30,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=100,
    report_to="none",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [18]:
import numpy as np
from evaluate import load
accuracy_metric = load("accuracy")
f1_metric = load("f1")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"   # weighted vì imbalance
    )["f1"]

    return {"accuracy": acc, "f1_weighted": f1}

In [27]:
from transformers import Trainer
import torch
class WeightedTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [29]:
import os
from transformers import AutoModelForSequenceClassification
output_dir = training_args.output_dir

# Tìm checkpoint
checkpoints = [
    d for d in os.listdir(output_dir)
    if d.startswith("checkpoint-") and d.split("-")[-1].isdigit()
]

if checkpoints:
    # Sắp xếp theo số epoch/step
    latest_checkpoint = max(checkpoints, key=lambda x: int(x.split("-")[-1]))
    model_path = os.path.join(output_dir, latest_checkpoint)
    print(f"Load từ checkpoint tốt nhất: {model_path}")
else:
    model_path = "vinai/phobert-base-v2"
    print("Không tìm thấy checkpoint → load base model (có thể chưa train đầy đủ)")
#load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=len(unique_classes),
    ignore_mismatched_sizes=True
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

# Đánh giá trên test set
print("\nĐánh giá trên test set...")
test_results = trainer.evaluate(tokenized_datasets["test"])

print("\nKết quả đánh giá:")
for k, v in test_results.items():
    if isinstance(v, float):
        print(f"  {k:<25}: {v:.4f}")
    else:
        print(f"  {k:<25}: {v}")

final_dir = "./phobert-3emotion-final"
os.makedirs(final_dir, exist_ok=True)

trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"\nModel cuối cùng đã được lưu tại: {final_dir}")

Không tìm thấy checkpoint → load base model (có thể chưa train đầy đủ)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Đánh giá trên test set...



Kết quả đánh giá:
  eval_loss                : 1.0883
  eval_model_preparation_time: 0.0253
  eval_accuracy            : 0.5333
  eval_f1_weighted         : 0.4320
  eval_runtime             : 2.2314
  eval_samples_per_second  : 201.6680
  eval_steps_per_second    : 12.9960


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model cuối cùng đã được lưu tại: ./phobert-3emotion-final
